# Find the best visual attempt

In this guide, you'll run a few variations of one task and keep the best result in a few lines.
The sample task is an infographic prompt. The same pattern works for bug fixes, UI directions,
prompts, queries, migrations, and other work where stronger alternatives are worth comparing.

## Setup

Load the launch helpers.

In [ ]:
import pathlib
import sys


def _find_visual_artifact_example_root():
    cwd = pathlib.Path.cwd().resolve()
    candidates = []
    for base in (cwd, *cwd.parents):
        candidates.append(base)
        candidates.append(base / "examples" / "notebooks" / "visual_artifact")
    for candidate in candidates:
        if (candidate / "shepherd_usecases" / "visual_artifact" / "launch.py").exists():
            return candidate
    raise RuntimeError(
        "Cannot find examples/notebooks/visual_artifact. "
        "Launch JupyterLab from the repository root with `make notebooks`."
    )


example_root = _find_visual_artifact_example_root()
if str(example_root) not in sys.path:
    sys.path.insert(0, str(example_root))
try:
    from shepherd_usecases.visual_artifact import launch, viz
except Exception as exc:
    raise RuntimeError(
        "Could not import the visual-artifact notebook helpers. "
        "Launch JupyterLab from the repository root with `make notebooks`."
    ) from exc

launch.bootstrap(example_root=example_root)

## What this run does

Variant Studio is a best-of-N run: it turns one prompt into several isolated attempts, each in its
own run, then hands the finished attempts to a reviewer that keeps the strongest.

## The input

For this example, the run starts with one prompt and two short variant instructions.

In [ ]:
prompt = launch.default_prompt()
variants = launch.variant_prompts()
{"prompt": prompt, "variants": variants}

## Run Shepherd

First, open a workspace for the prompt.

In [ ]:
workspace = launch.open_workspace("visual-variant-studio", prompt=prompt, metadata={"usecase": "visual-variant-studio"})

Now run one attempt per variant from that workspace. Each run is isolated, so the two attempts
write their artifacts without seeing each other.

In [ ]:
attempts = {}
for variant, instruction in variants.items():
    attempts[variant] = launch.run_static(
        workspace,
        name=variant,
        output_path=launch.ARTIFACT_PATH,
        output_text=launch.variant_html(prompt, variant),
        metadata={"variant": variant, "instruction": instruction},
    )

viz.show(viz.run_artifacts(attempts))

With both attempts finished, run the reviewer task. It receives the completed attempt runs as
artifact references — `candidate_refs` — so it can read each one's artifact and pick the best.

In [ ]:
candidate_refs = {
    f"candidate_{name}": launch.artifact_ref(run, label=name)
    for name, run in attempts.items()
}
reviewer = launch.run_with_artifact_refs(
    workspace,
    name="review",
    refs=candidate_refs,
    output_path=launch.VERDICT_PATH,
    output_content=launch.review_content(prompt, attempts),
    after=list(attempts.values()),
)
selection = launch.selection_from_review(reviewer)
viz.show(viz.review_summary(selection.candidates, selected=selection.selected))

In [ ]:
for name, run in attempts.items():
    if name == selection.selected:
        run.output().select()
    else:
        run.output().discard()
reviewer.output().release()

## Trace

How does Shepherd keep the attempts isolated and let one task read another's work? The trace below
captures every action an agent took while executing the tasks above, plus the
relationships among those actions. Click the nodes to learn more about the events.

In [ ]:
notes = {run.run_ref: name for name, run in attempts.items()}
notes[reviewer.run_ref] = f"review: selected {selection.selected}"
viz.show(viz.trace(workspace.flow, {**attempts, "review": reviewer}, notes=notes, height="620px", detail="events"))

## Optional Live Claude Run

The default path above is deterministic. Set the flag below only on a machine with the local Claude CLI,
credentials, and native jail support.

In [ ]:
RUN_LIVE_CLAUDE = False
CLAUDE_MODEL = None

if RUN_LIVE_CLAUDE:
    launch.require_claude()
    live_workspace = launch.open_workspace("visual-variant-studio-live", prompt=prompt, metadata={"usecase": "uc1-live"})
    try:
        live_attempts = {}
        for variant, instruction in variants.items():
            live_attempts[variant] = launch.run_claude_artifact(
                live_workspace,
                name=f"live-{variant}",
                prompt=prompt,
                variant=variant,
                instruction=instruction,
                model=CLAUDE_MODEL,
                metadata={"variant": variant, "mode": "live-claude"},
            )
        live_refs = {
            f"candidate_{name}": launch.artifact_ref(run, label=name)
            for name, run in live_attempts.items()
        }
        live_reviewer = launch.run_claude_review(
            live_workspace,
            name="live-review",
            prompt=prompt,
            refs=live_refs,
            after=list(live_attempts.values()),
            model=CLAUDE_MODEL,
            metadata={"mode": "live-claude"},
        )
        live_selection = launch.selection_from_review(live_reviewer)
        viz.show(viz.review_summary(live_selection.candidates, selected=live_selection.selected))
        for name, run in live_attempts.items():
            if name == live_selection.selected:
                run.output().select()
            else:
                run.output().discard()
        live_reviewer.output().release()
        viz.show(viz.trace(live_workspace.flow, {**live_attempts, "review": live_reviewer}, height="620px", detail="events"))
    finally:
        live_workspace.close()
else:
    launch.claude_preflight()

In [ ]:
workspace.close()

## Want to understand how it works?

If you want to understand how this works in greater detail, open the
[Variant Studio internals notebook](visual_variant_studio_internals.ipynb).

## Next steps

- [Find the cheapest model that still does the job](model_right_sizing_lab.ipynb)
- [Recover a pipeline when one step drifts](visual_pipeline_recovery.ipynb)